# 02 · Kokoro TTS：82M 参数的开源语音合成

**硬件**：🟢 CPU 可跑（模型仅 82M 参数 / ~330MB，笔记本实时合成无压力）

## 本 notebook 你将学到

1. 跑通 Kokoro-82M：TTS Arena 顶级、Apache 2.0、CPU 实时——开源 TTS 的"性价比之王"
2. 理解它的技术路线：**非自回归**（StyleTTS2 风格）vs 06 章理论里的 codec LM 路线，各自的取舍
3. 多语言、多音色、语速控制
4. 用 Whisper 做"回环测试"（TTS → ASR），自动化验证合成质量

## 前置依赖

部分语言的音素化依赖 espeak-ng：
- macOS: `brew install espeak-ng`
- Ubuntu: `sudo apt-get install espeak-ng`
- 中文支持: `pip install "misaki[zh]"`

In [ ]:
%pip install -q "kokoro>=0.9.4" soundfile
# 需要中文时再装: %pip install -q "misaki[zh]"

## 1. 英文合成

`KPipeline` 的 `lang_code` 决定音素化前端：`'a'`=美式英语，`'b'`=英式，`'z'`=中文，`'j'`=日语。

Kokoro 是**非自回归**架构：文本 → 音素 → 时长预测 → 一次性生成整段波形。好处是快且稳（不会像自回归模型那样偶尔复读/漏字），代价是没有 codec LM 路线的零样本克隆和上下文表现力——对照 [theory.md](../theory.md) 第 3 节体会这组 trade-off。

In [ ]:
from kokoro import KPipeline
from IPython.display import Audio as AudioPlayer, display
import soundfile as sf
import numpy as np

pipe_en = KPipeline(lang_code="a")  # American English

text_en = (
    "Kokoro is an open source text to speech model with only eighty two million parameters. "
    "It runs in real time on a laptop CPU, and it is licensed under Apache two point zero."
)

chunks = []
for gs, ps, audio in pipe_en(text_en, voice="af_heart"):
    print(f"文本片段: {gs[:60]}..." if len(gs) > 60 else f"文本片段: {gs}")
    print(f"音素: {ps[:80]}")
    chunks.append(audio)

wav_en = np.concatenate(chunks)
sf.write("kokoro_en.wav", wav_en, 24000)
display(AudioPlayer(wav_en, rate=24000))

## 2. 音色与语速

Kokoro 内置 54 个音色（voice embedding，不是不同模型）。命名规则：`af_*` 美式女声、`am_*` 美式男声、`bf_*/bm_*` 英式、`zf_*/zm_*` 中文。

In [ ]:
demo = "The same sentence, three different voices."

for voice in ["af_heart", "af_bella", "am_michael"]:
    audio = np.concatenate([a for _, _, a in pipe_en(demo, voice=voice)])
    print(f"voice = {voice}")
    display(AudioPlayer(audio, rate=24000))

# 语速控制
for speed in [0.8, 1.0, 1.4]:
    audio = np.concatenate([a for _, _, a in pipe_en(demo, voice="af_heart", speed=speed)])
    print(f"speed = {speed}")
    display(AudioPlayer(audio, rate=24000))

## 3. 中文合成

需要先 `pip install "misaki[zh]"`（中文 G2P 前端）。中文 TTS 的难点在多音字（"重庆"的重 vs "重要"的重）——G2P 前端的质量直接决定上限。

In [ ]:
try:
    pipe_zh = KPipeline(lang_code="z")
    text_zh = "多模态一零一是一个兼具理论与实践的开源教程，重点是把原理学清楚。"
    wav_zh = np.concatenate([a for _, _, a in pipe_zh(text_zh, voice="zf_xiaobei")])
    sf.write("kokoro_zh.wav", wav_zh, 24000)
    display(AudioPlayer(wav_zh, rate=24000))
except Exception as e:
    print(f"中文管线初始化失败（大概率缺 misaki[zh]）: {e}")

## 4. 回环测试：TTS → ASR 自动验证

怎么不靠耳朵批量验证 TTS 质量？把合成音频喂回 Whisper，对比转写和原文——**WER 高说明发音有问题**。这是生产中做 TTS 回归测试的常用技巧（当然它测不出音色/自然度，那些仍需人评或 Audio Turing Test 类基准）。

In [ ]:
%pip install -q transformers jiwer
import jiwer
from transformers import pipeline as hf_pipeline

asr = hf_pipeline("automatic-speech-recognition", model="openai/whisper-small")
hyp = asr("kokoro_en.wav")["text"]

norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                      jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
wer = jiwer.wer(norm(text_en), norm(hyp))
print(f"原文:   {text_en}")
print(f"ASR 回读: {hyp}")
print(f"回环 WER: {wer:.2%}")

## 练习

1. 数字、缩写、URL 是 TTS 的经典坑：让 Kokoro 读 `"GPT-5 costs $1.25 per 1M tokens, see https://example.com"`，听听哪里翻车，思考文本归一化（text normalization）该做什么。
2. 写个脚本把 [theory.md](../theory.md) 整篇转成 podcast 音频（注意长文本分段）。
3. 对比实验：同一段文本用 Kokoro 和 [03_higgs_v3_clone.ipynb](.)（计划中）的 Higgs Audio v3 各合成一遍，从自然度/表现力/速度三个维度打分——体会非自回归 vs codec LM 两条路线的差异。

**下一站**：`04_voice_pipeline.ipynb`（计划中）— 把 Whisper + LLM + Kokoro 串成完整语音对话。